In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import quantum_algorithms as qa

In [ ]:
# Experiment sets

shots = 2048
n_values = [1, 3, 5, 7, 9]
p_values = np.arange(0, 1.01, 0.05)

functions = {
    "constant_1": qa.deutsch_jozsa.f_constant_1,
    "constant_0": qa.deutsch_jozsa.f_constant_0,
    "balanced_parity": qa.deutsch_jozsa.f_balanced_parity,
}

error_positions = {
    "D(P)1_before_H": qa.depolarizing.deutsch_jozsa_depolarizing1,
    "D(P)2_after_first_H": qa.depolarizing.deutsch_jozsa_depolarizing2,
    "D(P)3_after_oracle": qa.depolarizing.deutsch_jozsa_depolarizing3,
    "D(P)4_after_final_H": qa.depolarizing.deutsch_jozsa_depolarizing4,
}

In [ ]:
# Run simulation with shots
results = []

for n in n_values:
    print("Running n =", n)

    target_qubit = n // 2

    for function_name, f in functions.items():

        state_ref = qa.deutsch_jozsa.deutsch_jozsa(n, f)

        probs_ref = qa.deutsch_jozsa.measure_probs_first_n(state_ref, n)
        probs_ref = qa.depolarizing.clean_probs(probs_ref)

        samples_ref = qa.deutsch_jozsa.sample_probs(probs_ref, shots)
        P0_ref = samples_ref[0] / shots

        if function_name.startswith("constant"):
            success_ref = P0_ref
        else:
            success_ref = 1 - P0_ref

        for p in p_values:

            results.append({
                "n": n,
                "p": p,
                "target_qubit": target_qubit,
                "function": function_name,
                "error_position": "no_error",
                "P0": P0_ref,
                "success": success_ref,
                "shots": shots,
            })

            for error_name, error_function in error_positions.items():

                rho_error = error_function(n, f, p, target_qubit)

                probs_error = qa.depolarizing.measure_probs_first_n_rho(
                    rho_error, n
                )
                probs_error = qa.depolarizing.clean_probs(probs_error)

                samples_error = qa.deutsch_jozsa.sample_probs(
                    probs_error, shots
                )

                P0_error = samples_error[0] / shots

                if function_name.startswith("constant"):
                    success_error = P0_error
                else:
                    success_error = 1 - P0_error

                results.append({
                    "n": n,
                    "p": p,
                    "target_qubit": target_qubit,
                    "function": function_name,
                    "error_position": error_name,
                    "P0": P0_error,
                    "success": success_error,
                    "shots": shots,
                })

df_dep = pd.DataFrame(results)

df_dep.head()

In [ ]:
df_dep.to_csv("depolarizing_noise_effect_results_shots_2048.csv", index=False)

In [ ]:
qa.plotting.depolarizing_noise_effect(df_dep, 9, "constant_1", shots)
qa.plotting.depolarizing_noise_effect(df_dep, 9, "constant_0", shots)
qa.plotting.depolarizing_noise_effect(df_dep, 9,"balanced_parity", shots)

In [ ]:
qa.plotting.depolarizing_noise_sensitivity(df_dep, 9, "constant_1", shots)
qa.plotting.depolarizing_noise_sensitivity(df_dep, 9, "constant_0", shots)
qa.plotting.depolarizing_noise_sensitivity(df_dep, 9, "balanced_parity", shots)